# Prisoner's Dilemma — Results Analysis
**ESADE MIBA Capstone 2026 · Strategic Coherence in LLMs**

This notebook reproduces and extends the results shown in `prisoners_dilemma_report_v4.html`  
with no hardcoded conclusions — every number is computed from raw data with 95% confidence intervals.

**Games**: 4 conditions × ~10 runs each → 80 CSV files  
**Conditions**: `undisclosed` · `ai` · `human` · `human_prior`  
**Models**: Claude Opus · Claude Sonnet · GPT-4o · GPT-4o-mini · Gemini Flash · Gemini Pro  
**Rounds per game**: 20  
**Temperature**: 0.6

---
### Sections
1. Setup & Data Loading
2. Data Quality Check
3. Framing Effect: COOPERATE/DEFECT vs EXPAND/HOLD
4. Overall Cooperation Rate by Condition (with 95% CI)
5. Per-Model Cooperation Rate by Condition
6. Round-by-Round Dynamics (Early / Mid / Late phases)
7. Strategic Metrics: ρ, β, TfT Adherence, Backward Induction
8. Matchup Heatmap
9. Statistical Tests (Kruskal-Wallis + post-hoc)
10. Key Findings Summary

## 1. Setup & Data Loading

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, bootstrap
from statsmodels.stats.multitest import multipletests
import itertools

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# ── Path configuration ────────────────────────────────────────────────────────
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR  = os.path.join(REPO_ROOT, 'all_raw')

print(f'Repo root : {REPO_ROOT}')
print(f'Data dir  : {DATA_DIR}')
print(f'Exists    : {os.path.isdir(DATA_DIR)}')

In [ ]:
# ── Load all PD CSVs ──────────────────────────────────────────────────────────
pd_files = sorted(glob.glob(os.path.join(DATA_DIR, 'pd_results_*.csv')))
print(f'Found {len(pd_files)} PD CSV files')

frames = []
for f in pd_files:
    df_tmp = pd.read_csv(f)
    df_tmp.columns = df_tmp.columns.str.strip()
    df_tmp['source_file'] = os.path.basename(f)
    # Detect framing variant from filename
    df_tmp['framing'] = 'HE' if '_HE_' in os.path.basename(f) else 'CD'
    frames.append(df_tmp)

raw = pd.concat(frames, ignore_index=True)
print(f'Total rows loaded : {len(raw):,}')
print(f'Conditions        : {sorted(raw["condition"].unique())}')
print(f'Framings          : {sorted(raw["framing"].unique())}')
raw.head(3)

## 2. Data Quality Check

In [ ]:
# ── Schema validation ─────────────────────────────────────────────────────────
required_cols = [
    'game_id','condition','matchup','round',
    'model_a','action_a','belief_a','payoff_a','cumulative_a',
    'model_b','action_b','belief_b','payoff_b','cumulative_b',
    'temperature_a','prompt_version','timestamp'
]
missing = [c for c in required_cols if c not in raw.columns]
assert not missing, f'Missing columns: {missing}'

# ── Action validation ─────────────────────────────────────────────────────────
valid_actions = {'C', 'D'}
bad_a = ~raw['action_a'].isin(valid_actions)
bad_b = ~raw['action_b'].isin(valid_actions)
print(f'Invalid action_a : {bad_a.sum()} rows  ({bad_a.mean()*100:.1f}%)')
print(f'Invalid action_b : {bad_b.sum()} rows  ({bad_b.mean()*100:.1f}%)')

# ── Belief range validation ───────────────────────────────────────────────────
for col in ['belief_a', 'belief_b']:
    out = raw[(raw[col] < 0) | (raw[col] > 1)]
    print(f'{col} out of [0,1] : {len(out)} rows')

# ── Round counts ─────────────────────────────────────────────────────────────
round_counts = raw.groupby(['source_file'])['round'].max()
print(f'\nRound max per file:\n{round_counts.value_counts().to_string()}')

# ── Rows per condition + framing ──────────────────────────────────────────────
print('\nRows per condition × framing:')
print(raw.groupby(['condition','framing']).size().to_string())

In [ ]:
# ── Build long-form dataset (one row per player per round) ────────────────────
def to_long(df):
    def side(player):
        other = 'b' if player == 'a' else 'a'
        rt_col = f'response_time_{player}' if f'response_time_{player}' in df.columns else None
        base = df[[
            'game_id','condition','matchup','round','prompt_version',
            'framing','source_file',
            f'model_{player}', f'action_{player}', f'belief_{player}',
            f'payoff_{player}', f'cumulative_{player}',
            f'action_{other}', f'belief_{other}',
        ]].rename(columns={
            f'model_{player}':      'model',
            f'action_{player}':     'action',
            f'belief_{player}':     'belief',
            f'payoff_{player}':     'payoff',
            f'cumulative_{player}': 'cumulative',
            f'action_{other}':      'opp_action',
            f'belief_{other}':      'opp_belief',
        }).copy()
        if rt_col:
            base['response_time'] = df[rt_col].values
        base['side'] = player
        return base
    return pd.concat([side('a'), side('b')], ignore_index=True)

long = to_long(raw)
long['coop']     = (long['action']     == 'C').astype(int)
long['opp_coop'] = (long['opp_action'] == 'C').astype(float)
long['beta_err'] = (long['belief'] - long['opp_coop']).abs()
long['phase']    = pd.cut(long['round'], bins=[0,5,15,20],
                          labels=['early','mid','late'])

print(f'Long-form rows : {len(long):,}')
print(f'Unique models  : {sorted(long["model"].unique())}')
long.head(3)

## 3. Framing Effect: COOPERATE/DEFECT vs EXPAND/HOLD

In [ ]:
# ── Confidence interval helper ────────────────────────────────────────────────
def mean_ci(x, confidence=0.95):
    """Return (mean, lower_ci, upper_ci) using Wilson interval for proportions."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan
    m = x.mean()
    # For proportions (0/1): Wilson score interval
    if set(np.unique(x)).issubset({0.0, 1.0}):
        from statsmodels.stats.proportion import proportion_confint
        lo, hi = proportion_confint(int(x.sum()), n, alpha=1-confidence, method='wilson')
        return m, lo, hi
    # For continuous: t-interval
    se = stats.sem(x)
    t  = stats.t.ppf((1 + confidence) / 2, df=n-1)
    return m, m - t*se, m + t*se

# ── Cooperation rate by framing ───────────────────────────────────────────────
framing_stats = []
for framing, grp in long.groupby('framing'):
    m, lo, hi = mean_ci(grp['coop'])
    framing_stats.append({'framing': framing, 'mean': m, 'ci_lo': lo, 'ci_hi': hi, 'n': len(grp)})
framing_df = pd.DataFrame(framing_stats)

# Also test per condition
framing_cond = []
for (cond, framing), grp in long.groupby(['condition','framing']):
    m, lo, hi = mean_ci(grp['coop'])
    framing_cond.append({'condition': cond, 'framing': framing,
                         'mean': m, 'ci_lo': lo, 'ci_hi': hi, 'n': len(grp)})
framing_cond_df = pd.DataFrame(framing_cond)

# Statistical test: CD vs HE
cd_coop = long[long['framing']=='CD']['coop']
he_coop = long[long['framing']=='HE']['coop']
if len(cd_coop) > 0 and len(he_coop) > 0:
    stat, pval = mannwhitneyu(cd_coop, he_coop, alternative='two-sided')
    print(f'Mann-Whitney U (CD vs HE): U={stat:.0f}, p={pval:.4f}')

print(framing_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: overall framing comparison
ax = axes[0]
colors = ['#5b8dee','#e8865a']
for i, row in framing_df.iterrows():
    label = 'COOPERATE/DEFECT' if row['framing']=='CD' else 'EXPAND/HOLD'
    ax.bar(i, row['mean'], color=colors[i], alpha=0.85, width=0.5, label=label)
    ax.errorbar(i, row['mean'],
                yerr=[[row['mean']-row['ci_lo']], [row['ci_hi']-row['mean']]],
                fmt='none', color='black', capsize=6, linewidth=1.5)
    ax.text(i, row['mean'] + 0.03, f"{row['mean']:.1%}", ha='center', fontsize=11, fontweight='bold')
ax.set_xticks([0,1])
ax.set_xticklabels(['CD\n(COOPERATE/DEFECT)', 'HE\n(EXPAND/HOLD)'])
ax.set_ylabel('Cooperation Rate')
ax.set_ylim(0, 1)
ax.set_title('Cooperation Rate by Action Framing\n(error bars = 95% CI, Wilson)')

# Right: framing × condition
ax = axes[1]
if len(framing_cond_df) > 0:
    conds = sorted(framing_cond_df['condition'].unique())
    framings = ['CD','HE']
    x = np.arange(len(conds))
    width = 0.35
    for j, framing in enumerate(framings):
        sub = framing_cond_df[framing_cond_df['framing']==framing].set_index('condition').reindex(conds)
        if sub['mean'].notna().any():
            offset = (j - 0.5) * width
            ax.bar(x + offset, sub['mean'].fillna(0), width, color=colors[j], alpha=0.8,
                   label='CD' if framing=='CD' else 'HE')
            ax.errorbar(x + offset, sub['mean'].fillna(0),
                        yerr=[(sub['mean']-sub['ci_lo']).fillna(0),
                              (sub['ci_hi']-sub['mean']).fillna(0)],
                        fmt='none', color='black', capsize=4, linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(conds, rotation=15)
    ax.set_ylabel('Cooperation Rate')
    ax.set_ylim(0, 1)
    ax.set_title('Framing Effect by Condition\n(error bars = 95% CI)')
    ax.legend(title='Framing')

plt.tight_layout()
plt.savefig('../notebooks/fig_framing_effect.png', bbox_inches='tight')
plt.show()
print('Saved: fig_framing_effect.png')

## 4. Overall Cooperation Rate by Condition (with 95% CI)

In [ ]:
# ── Aggregate at game level first to avoid inflating N ────────────────────────
# Unit of analysis: one observation per complete game (per model per run)
game_level = (
    long.groupby(['source_file','game_id','condition','framing','model'])
    .agg(coop_rate=('coop','mean'), n_rounds=('round','count'))
    .reset_index()
)

# ── Condition-level summary ───────────────────────────────────────────────────
cond_stats = []
for (cond, framing), grp in game_level.groupby(['condition','framing']):
    m, lo, hi = mean_ci(grp['coop_rate'])
    cond_stats.append({
        'condition': cond, 'framing': framing,
        'mean': m, 'ci_lo': lo, 'ci_hi': hi,
        'n_games': len(grp), 'n_rounds': int(grp['n_rounds'].sum())
    })
cond_df = pd.DataFrame(cond_stats).sort_values(['condition','framing'])

# Pretty print
print('=== Cooperation Rate by Condition (game-level aggregation) ===')
for _, row in cond_df.iterrows():
    print(f"  {row['condition']:15s} [{row['framing']}]  "
          f"{row['mean']:.1%}  95% CI [{row['ci_lo']:.1%}, {row['ci_hi']:.1%}]  "
          f"({row['n_games']} games, {row['n_rounds']} rounds)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

palette = {'CD': '#5b8dee', 'HE': '#e8865a'}
conds = sorted(cond_df['condition'].unique())
framings = sorted(cond_df['framing'].unique())
x = np.arange(len(conds))
n_framings = len(framings)
width = 0.35

for j, framing in enumerate(framings):
    sub = cond_df[cond_df['framing']==framing].set_index('condition').reindex(conds)
    offset = (j - (n_framings-1)/2) * width
    bars = ax.bar(x + offset, sub['mean'].fillna(0), width,
                  color=palette.get(framing, '#888'), alpha=0.85,
                  label=f'{framing} framing')
    ax.errorbar(
        x + offset, sub['mean'].fillna(0),
        yerr=[(sub['mean']-sub['ci_lo']).fillna(0),
              (sub['ci_hi']-sub['mean']).fillna(0)],
        fmt='none', color='#222', capsize=5, linewidth=1.5
    )
    for xi, (_, row) in zip(x + offset, sub.iterrows()):
        if not np.isnan(row['mean']):
            ax.text(xi, row['mean'] + 0.025, f"{row['mean']:.1%}",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(conds)
ax.set_ylabel('Cooperation Rate')
ax.set_ylim(0, 1.05)
ax.set_title('Overall Cooperation Rate by Condition\n(error bars = 95% Wilson CI, game-level aggregation)')
ax.legend(title='Action framing')
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5, label='50% baseline')

plt.tight_layout()
plt.savefig('../notebooks/fig_coop_by_condition.png', bbox_inches='tight')
plt.show()

## 5. Per-Model Cooperation Rate by Condition

In [ ]:
model_cond_stats = []
for (model, cond, framing), grp in game_level.groupby(['model','condition','framing']):
    m, lo, hi = mean_ci(grp['coop_rate'])
    model_cond_stats.append({
        'model': model, 'condition': cond, 'framing': framing,
        'mean': m, 'ci_lo': lo, 'ci_hi': hi, 'n_games': len(grp)
    })
model_cond_df = pd.DataFrame(model_cond_stats)

MODEL_COLORS = {
    'Claude Opus':   '#8b5cf6',
    'Claude Sonnet': '#a78bfa',
    'GPT-4o':        '#10b981',
    'Gpt4O':         '#10b981',
    'GPT-4o Mini':   '#34d399',
    'Gpt4O Mini':    '#34d399',
    'Gemini Flash':  '#60a5fa',
    'Gemini Pro':    '#93c5fd',
}
def mcolor(name):
    n = name.lower()
    if 'claude' in n: return '#8b5cf6'
    if 'gpt'   in n: return '#10b981'
    if 'gemini'in n: return '#60a5fa'
    return '#888'

models  = sorted(model_cond_df['model'].unique())
conds   = sorted(model_cond_df['condition'].unique())
framings = sorted(model_cond_df['framing'].unique())

print(f'Models in dataset: {models}')
model_cond_df.head()

In [ ]:
for framing in framings:
    sub = model_cond_df[model_cond_df['framing']==framing]
    if sub.empty:
        continue

    pivot = sub.pivot(index='model', columns='condition', values='mean')
    pivot_lo = sub.pivot(index='model', columns='condition', values='ci_lo')
    pivot_hi = sub.pivot(index='model', columns='condition', values='ci_hi')

    cond_cols = [c for c in conds if c in pivot.columns]
    n_conds = len(cond_cols)
    pivot = pivot[cond_cols]
    pivot_lo = pivot_lo[cond_cols]
    pivot_hi = pivot_hi[cond_cols]

    model_list = pivot.index.tolist()
    x = np.arange(len(model_list))
    width = 0.7 / n_conds
    cond_colors = {'undisclosed':'#94a3b8','ai':'#5b8dee','human':'#52c97a','human_prior':'#e8865a'}

    fig, ax = plt.subplots(figsize=(12, 5))
    for j, cond in enumerate(cond_cols):
        offset = (j - (n_conds-1)/2) * width
        vals   = pivot[cond].values
        lo_err = (pivot[cond] - pivot_lo[cond]).values
        hi_err = (pivot_hi[cond] - pivot[cond]).values
        ax.bar(x + offset, np.nan_to_num(vals), width,
               color=cond_colors.get(cond,'#aaa'), alpha=0.85, label=cond)
        ax.errorbar(x + offset, np.nan_to_num(vals),
                    yerr=[np.nan_to_num(lo_err), np.nan_to_num(hi_err)],
                    fmt='none', color='#222', capsize=4, linewidth=1.2)

    ax.set_xticks(x)
    ax.set_xticklabels(model_list, rotation=20, ha='right')
    ax.set_ylabel('Cooperation Rate')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'Per-Model Cooperation Rate by Condition — {framing} framing\n(error bars = 95% Wilson CI)')
    ax.legend(title='Condition', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'../notebooks/fig_model_coop_{framing}.png', bbox_inches='tight')
    plt.show()

## 6. Round-by-Round Dynamics (Early / Mid / Late)

In [ ]:
# ── Cooperation over rounds, averaged across all games per condition ───────────
round_stats = []
for (cond, framing, rnd), grp in long.groupby(['condition','framing','round']):
    m, lo, hi = mean_ci(grp['coop'])
    round_stats.append({'condition':cond,'framing':framing,'round':rnd,
                        'mean':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
round_df = pd.DataFrame(round_stats)

cond_colors = {'undisclosed':'#94a3b8','ai':'#5b8dee','human':'#52c97a','human_prior':'#e8865a'}

for framing in framings:
    sub = round_df[round_df['framing']==framing]
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(12, 5))
    for cond in sorted(sub['condition'].unique()):
        s = sub[sub['condition']==cond].sort_values('round')
        color = cond_colors.get(cond,'#aaa')
        ax.plot(s['round'], s['mean'], marker='o', markersize=4,
                label=cond, color=color, linewidth=1.8)
        ax.fill_between(s['round'], s['ci_lo'], s['ci_hi'],
                        color=color, alpha=0.12)
    # Phase boundaries
    ax.axvline(5,  color='grey', linestyle=':', linewidth=1, alpha=0.6)
    ax.axvline(15, color='grey', linestyle=':', linewidth=1, alpha=0.6)
    ax.text(2.5,  0.02, 'Early',  ha='center', color='grey', fontsize=9)
    ax.text(10,   0.02, 'Mid',    ha='center', color='grey', fontsize=9)
    ax.text(17.5, 0.02, 'Late',   ha='center', color='grey', fontsize=9)
    ax.set_xlabel('Round')
    ax.set_ylabel('Cooperation Rate')
    ax.set_ylim(0, 1)
    ax.set_xlim(1, 20)
    ax.set_title(f'Cooperation Over Rounds by Condition — {framing} framing\n(shaded = 95% CI)')
    ax.legend(title='Condition')
    plt.tight_layout()
    plt.savefig(f'../notebooks/fig_round_dynamics_{framing}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Phase-level summary table ─────────────────────────────────────────────────
phase_stats = []
for (cond, framing, phase), grp in long.groupby(['condition','framing','phase'], observed=True):
    m, lo, hi = mean_ci(grp['coop'])
    phase_stats.append({'condition':cond,'framing':framing,'phase':str(phase),
                        'mean':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
phase_df = pd.DataFrame(phase_stats)

print('=== Cooperation by Phase ===')
for _, row in phase_df.sort_values(['condition','framing','phase']).iterrows():
    print(f"  {row['condition']:15s} [{row['framing']}] {row['phase']:6s}  "
          f"{row['mean']:.1%}  95% CI [{row['ci_lo']:.1%}, {row['ci_hi']:.1%}]")

## 7. Strategic Metrics: ρ, β, TfT Adherence, Backward Induction

In [ ]:
# ── ρ  — Conditional reciprocity ─────────────────────────────────────────────
# P(C | opponent C at t-1) - P(C | opponent D at t-1)
ls = long.sort_values(['source_file','game_id','model','round']).copy()
ls['prev_opp'] = ls.groupby(['source_file','game_id','model'])['opp_action'].shift(1)
ls_valid = ls.dropna(subset=['prev_opp'])

rho_rows = []
for (model, cond, framing), grp in ls_valid.groupby(['model','condition','framing']):
    cc = grp[grp['prev_opp']=='C']['coop']
    cd = grp[grp['prev_opp']=='D']['coop']
    if len(cc) == 0 or len(cd) == 0:
        continue
    rho = cc.mean() - cd.mean()
    # CI via bootstrap on the difference
    rng = np.random.default_rng(42)
    diffs = []
    for _ in range(1000):
        s_cc = rng.choice(cc.values, size=len(cc), replace=True)
        s_cd = rng.choice(cd.values, size=len(cd), replace=True)
        diffs.append(s_cc.mean() - s_cd.mean())
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    rho_rows.append({'model':model,'condition':cond,'framing':framing,
                     'rho':rho,'ci_lo':lo,'ci_hi':hi,
                     'n_c':len(cc),'n_d':len(cd)})

rho_df = pd.DataFrame(rho_rows)
print('=== ρ (Conditional Reciprocity) ===')
print(rho_df[['model','condition','framing','rho','ci_lo','ci_hi']].sort_values(['condition','model']).to_string(index=False))

In [ ]:
# ── β  — Belief calibration ───────────────────────────────────────────────────
beta_rows = []
for (model, cond, framing), grp in long.groupby(['model','condition','framing']):
    m, lo, hi = mean_ci(grp['beta_err'])
    beta_rows.append({'model':model,'condition':cond,'framing':framing,
                      'beta':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
beta_df = pd.DataFrame(beta_rows)

print('=== β (Belief Calibration — MAE) ===')
print(beta_df[['model','condition','framing','beta','ci_lo','ci_hi']].sort_values(['condition','model']).to_string(index=False))

In [ ]:
# ── TfT Adherence — P(action_t == opp_action_{t-1}) ─────────────────────────
ls2 = long.sort_values(['source_file','game_id','model','round']).copy()
ls2['prev_opp_coop'] = ls2.groupby(['source_file','game_id','model'])['opp_coop'].shift(1)
tft_valid = ls2.dropna(subset=['prev_opp_coop']).copy()
tft_valid['tft_match'] = (tft_valid['coop'] == tft_valid['prev_opp_coop'].astype(int)).astype(float)

tft_rows = []
for (model, cond, framing), grp in tft_valid.groupby(['model','condition','framing']):
    m, lo, hi = mean_ci(grp['tft_match'])
    tft_rows.append({'model':model,'condition':cond,'framing':framing,
                     'tft':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
tft_df = pd.DataFrame(tft_rows)

# ── Backward Induction Index — P(defect | round >= 18) ──────────────────────
bi_rows = []
late = long[long['round'] >= 18]
for (model, cond, framing), grp in late.groupby(['model','condition','framing']):
    defect_rate = 1.0 - grp['coop'].mean()
    _, lo, hi = mean_ci(1 - grp['coop'])   # CI on defection
    bi_rows.append({'model':model,'condition':cond,'framing':framing,
                    'bi':defect_rate,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
bi_df = pd.DataFrame(bi_rows)

print('=== TfT Adherence (sample) ===')
print(tft_df[['model','condition','framing','tft','ci_lo','ci_hi']].head(10).to_string(index=False))
print('\n=== Backward Induction Index (sample) ===')
print(bi_df[['model','condition','framing','bi','ci_lo','ci_hi']].head(10).to_string(index=False))

In [ ]:
# ── Strategic metrics summary plot ────────────────────────────────────────────
# Aggregate across conditions for a clean per-model view
metrics_summary = []
for model in sorted(long['model'].unique()):
    sub = long[long['model']==model]
    m_coop, lo_coop, hi_coop = mean_ci(sub['coop'])
    m_beta, lo_beta, hi_beta = mean_ci(sub['beta_err'])

    rho_sub = rho_df[rho_df['model']==model]
    m_rho = rho_sub['rho'].mean() if len(rho_sub) else np.nan

    tft_sub = tft_df[tft_df['model']==model]
    m_tft, lo_tft, hi_tft = mean_ci(tft_sub['tft'].dropna()) if len(tft_sub) else (np.nan,np.nan,np.nan)

    bi_sub = bi_df[bi_df['model']==model]
    m_bi, lo_bi, hi_bi = mean_ci(bi_sub['bi'].dropna()) if len(bi_sub) else (np.nan,np.nan,np.nan)

    metrics_summary.append({
        'model': model,
        'coop_rate': m_coop, 'coop_lo': lo_coop, 'coop_hi': hi_coop,
        'beta':      m_beta,  'beta_lo': lo_beta,  'beta_hi': hi_beta,
        'rho':       m_rho,
        'tft':       m_tft,   'tft_lo':  lo_tft,   'tft_hi':  hi_tft,
        'bi':        m_bi,    'bi_lo':   lo_bi,    'bi_hi':   hi_bi,
    })
metrics_df = pd.DataFrame(metrics_summary)
print(metrics_df[['model','coop_rate','rho','beta','tft','bi']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metric_specs = [
    ('coop_rate','coop_lo','coop_hi', 'Cooperation Rate',         axes[0,0]),
    ('rho',      None,      None,     'ρ (Conditional Reciprocity)', axes[0,1]),
    ('beta',     'beta_lo', 'beta_hi','β (Belief Calibration MAE)',  axes[1,0]),
    ('tft',      'tft_lo',  'tft_hi', 'TfT Adherence',              axes[1,1]),
]

model_list = metrics_df['model'].tolist()
colors_list = [mcolor(m) for m in model_list]

for val_col, lo_col, hi_col, title, ax in metric_specs:
    vals = metrics_df[val_col].values
    ax.barh(model_list, vals, color=colors_list, alpha=0.82)
    if lo_col and hi_col:
        xerr_lo = (metrics_df[val_col] - metrics_df[lo_col]).values
        xerr_hi = (metrics_df[hi_col]  - metrics_df[val_col]).values
        ax.errorbar(vals, model_list,
                    xerr=[np.nan_to_num(xerr_lo), np.nan_to_num(xerr_hi)],
                    fmt='none', color='#333', capsize=4, linewidth=1.3)
    ax.set_title(title)
    ax.set_xlabel('Value')
    ax.axvline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)

fig.suptitle('Strategic Profile — All Models (averaged across conditions)\n(error bars = 95% CI where applicable)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../notebooks/fig_strategic_metrics.png', bbox_inches='tight')
plt.show()

## 8. Matchup Heatmap

In [ ]:
# ── Cooperation rate of model_i vs model_j (who cooperates more against whom) ─
models_all = sorted(set(raw['model_a'].tolist() + raw['model_b'].tolist()))
idx = {m: i for i, m in enumerate(models_all)}
n = len(models_all)
mat = np.full((n, n), np.nan)
mat_lo = np.full((n, n), np.nan)
mat_hi = np.full((n, n), np.nan)

for matchup, grp in raw.groupby('matchup'):
    ma = grp['model_a'].iloc[0]
    mb = grp['model_b'].iloc[0]
    if ma not in idx or mb not in idx:
        continue
    coop_a = (grp['action_a'] == 'C').astype(float)
    coop_b = (grp['action_b'] == 'C').astype(float)
    ma_mean, ma_lo, ma_hi = mean_ci(coop_a)
    mb_mean, mb_lo, mb_hi = mean_ci(coop_b)
    mat[idx[ma]][idx[mb]] = ma_mean
    mat[idx[mb]][idx[ma]] = mb_mean
    mat_lo[idx[ma]][idx[mb]] = ma_lo
    mat_lo[idx[mb]][idx[ma]] = mb_lo
    mat_hi[idx[ma]][idx[mb]] = ma_hi
    mat_hi[idx[mb]][idx[ma]] = mb_hi

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(mat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Cooperation Rate')

for i in range(n):
    for j in range(n):
        if not np.isnan(mat[i,j]):
            ax.text(j, i, f"{mat[i,j]:.0%}",
                    ha='center', va='center', fontsize=8,
                    color='black' if 0.25 < mat[i,j] < 0.75 else 'white')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(models_all, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(models_all, fontsize=9)
ax.set_title('Cooperation Rate Heatmap — row model vs. column model\n(all conditions combined)')
plt.tight_layout()
plt.savefig('../notebooks/fig_matchup_heatmap.png', bbox_inches='tight')
plt.show()

## 9. Statistical Tests

In [ ]:
# ── Kruskal-Wallis: do conditions differ in cooperation rate? ─────────────────
# Unit: game-level cooperation rate (avoids round-level pseudo-replication)
print('=== Kruskal-Wallis: Cooperation Rate across Conditions ===')
for framing in framings:
    sub = game_level[game_level['framing']==framing]
    groups = [grp['coop_rate'].values for _, grp in sub.groupby('condition') if len(grp) >= 3]
    if len(groups) < 2:
        print(f'  [{framing}] Not enough groups')
        continue
    stat, p = kruskal(*groups)
    print(f'  [{framing}]  H={stat:.3f},  p={p:.4f}  (n_conditions={len(groups)})')

print()
# ── Kruskal-Wallis: do models differ in cooperation rate? ─────────────────────
print('=== Kruskal-Wallis: Cooperation Rate across Models ===')
for framing in framings:
    sub = game_level[game_level['framing']==framing]
    groups = [grp['coop_rate'].values for _, grp in sub.groupby('model') if len(grp) >= 3]
    if len(groups) < 2:
        continue
    stat, p = kruskal(*groups)
    print(f'  [{framing}]  H={stat:.3f},  p={p:.4f}  (n_models={len(groups)})')

In [ ]:
# ── Post-hoc pairwise Mann-Whitney U (conditions) with Holm-Bonferroni ────────
print('=== Pairwise Mann-Whitney U (conditions, Holm-Bonferroni corrected) ===')
for framing in framings:
    sub = game_level[game_level['framing']==framing]
    cond_list = sorted(sub['condition'].unique())
    pairs = list(itertools.combinations(cond_list, 2))
    if not pairs:
        continue
    raw_p, labels = [], []
    for c1, c2 in pairs:
        g1 = sub[sub['condition']==c1]['coop_rate'].dropna()
        g2 = sub[sub['condition']==c2]['coop_rate'].dropna()
        if len(g1) < 3 or len(g2) < 3:
            continue
        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
        raw_p.append(p)
        labels.append(f'{c1} vs {c2}')
    if not raw_p:
        continue
    reject, p_adj, _, _ = multipletests(raw_p, method='holm')
    print(f'\n  [{framing}]')
    for label, p_raw, p_c, rej in zip(labels, raw_p, p_adj, reject):
        sig = '***' if p_c < 0.001 else '**' if p_c < 0.01 else '*' if p_c < 0.05 else 'ns'
        print(f'    {label:35s}  p_raw={p_raw:.4f}  p_adj={p_c:.4f}  {sig}')

In [ ]:
# ── Post-hoc pairwise Mann-Whitney U (models) with Holm-Bonferroni ────────────
print('=== Pairwise Mann-Whitney U (models, Holm-Bonferroni corrected) ===')
for framing in framings:
    sub = game_level[game_level['framing']==framing]
    model_list_f = sorted(sub['model'].unique())
    pairs = list(itertools.combinations(model_list_f, 2))
    if not pairs:
        continue
    raw_p, labels = [], []
    for m1, m2 in pairs:
        g1 = sub[sub['model']==m1]['coop_rate'].dropna()
        g2 = sub[sub['model']==m2]['coop_rate'].dropna()
        if len(g1) < 3 or len(g2) < 3:
            continue
        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
        raw_p.append(p)
        labels.append(f'{m1} vs {m2}')
    if not raw_p:
        continue
    reject, p_adj, _, _ = multipletests(raw_p, method='holm')
    print(f'\n  [{framing}]')
    for label, p_raw, p_c, rej in zip(labels, raw_p, p_adj, reject):
        sig = '***' if p_c < 0.001 else '**' if p_c < 0.01 else '*' if p_c < 0.05 else 'ns'
        print(f'    {label:45s}  p_raw={p_raw:.4f}  p_adj={p_c:.4f}  {sig}')

## 10. Key Findings Summary

In [ ]:
# ── All numbers computed from data — nothing hardcoded ─────────────────────────
print('=' * 65)
print('KEY FINDINGS — Prisoner\'s Dilemma')
print('=' * 65)

print('\n[DATASET]')
print(f'  Total runs      : {len(pd_files)}')
print(f'  Total rounds    : {len(raw):,}')
print(f'  Conditions      : {", ".join(sorted(long["condition"].unique()))}')
print(f'  Framings        : {", ".join(sorted(long["framing"].unique()))}')
print(f'  Models          : {", ".join(sorted(long["model"].unique()))}')

print('\n[COOPERATION RATES BY CONDITION (all framings)]')
for (cond,), grp in game_level.groupby(['condition']):
    m, lo, hi = mean_ci(grp['coop_rate'])
    print(f'  {cond:15s}  {m:.1%}  95% CI [{lo:.1%}, {hi:.1%}]  (n={len(grp)} games)')

print('\n[FRAMING EFFECT]')
for (framing,), grp in game_level.groupby(['framing']):
    m, lo, hi = mean_ci(grp['coop_rate'])
    label = 'COOPERATE/DEFECT' if framing=='CD' else 'EXPAND/HOLD'
    print(f'  {label:20s}  {m:.1%}  95% CI [{lo:.1%}, {hi:.1%}]  (n={len(grp)} games)')

print('\n[STRATEGIC METRICS — averaged across all conditions]')
for _, row in metrics_df.iterrows():
    print(f"  {row['model']:20s}  "
          f"coop={row['coop_rate']:.1%}  "
          f"ρ={row['rho']:.3f}  "
          f"β={row['beta']:.3f}  "
          f"TfT={row['tft']:.1%}  "
          f"BI={row['bi']:.1%}")

print('\n[NOTE] All values are computed directly from raw CSVs.')
print('       95% CIs: Wilson interval for proportions, t-interval for continuous.')
print('       ρ CIs: bootstrap (n=1000).')
print('       Statistical tests: Kruskal-Wallis + Holm-Bonferroni corrected pairwise MW-U.')